In [0]:
from pyspark.sql.functions import current_timestamp, col

# In Databricks, create the database if you haven't already
spark.sql("CREATE DATABASE IF NOT EXISTS bronze_new")

BASE_PATH = 'gs://nawaf-etl-lake-dev-useast/raw'
SUBSETS = ['/crm', '/erp']




for subset in SUBSETS:
    path_to_read = BASE_PATH + subset
    print(f"--- Scanning directory: {path_to_read} ---")
    
    csvfiles = dbutils.fs.ls(path_to_read)
    
    for file in csvfiles:
        if file.name.endswith('.csv'):
            
            # 1. Read the CSV
            df = (spark.read
                .format('csv')
                .option('header', 'true')
                .option('inferSchema', 'true')
                .load(file.path)
            )
            
            # 2. Sanity Verification
            row_count = df.count()
            if row_count == 0:
                print(f"WARNING: {file.name} is empty. Skipping ingestion.")
                continue

            # --- THE FIX: Unity Catalog Metadata Injection ---
            df_with_metadata = (df
                .withColumn("_ingest_timestamp", current_timestamp())
                .withColumn("_source_file", col("_metadata.file_path")) 
            )
            # -----------------------------------------------

            prefix = subset.replace('/', '') 
            clean_filename = file.name[:-4]
            table_name = f'bronze_new.{prefix}_{clean_filename}'

            # 3. Write to Delta
            (df_with_metadata.write
                .format('delta')
                .mode('overwrite')
                .saveAsTable(table_name)
            )
            print(f'Successfully wrote {row_count} rows to: {table_name}\n')

    print(f"Finished processing {subset}.\n")